# 🎬 CogVideoX-5B-I2V (Ultra Reduzido para Colab)
Este notebook tenta carregar CogVideoX no Colab com o mínimo absoluto de uso de RAM e VRAM.
Limite: 8 frames, 16 passos de inferência.

In [ ]:
# ✅ Instalar dependências
!pip install diffusers==0.33.1 transformers accelerate einops gradio ffmpeg-python safetensors --quiet

In [ ]:
# 🔑 Login Hugging Face
from huggingface_hub import login
login()

In [ ]:
# 📦 Imports e configurações
import os
import torch
from PIL import Image
import gradio as gr
from diffusers import (
    CogVideoXImageToVideoPipeline,
    AutoencoderKLCogVideoX,
    CogVideoXTransformer3DModel,
)
from diffusers.utils import export_to_video, load_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs("outputs", exist_ok=True)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

In [ ]:
# 🚀 Carregar modelo com carregamento sob demanda (device_map='balanced')
from transformers import logging
logging.set_verbosity_error()

transformer = CogVideoXTransformer3DModel.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="transformer",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced"
)

text_encoder = AutoencoderKLCogVideoX.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="vae",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced"
)

pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    transformer=transformer,
    text_encoder=text_encoder,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced"
)

In [ ]:
# 🎥 Geração com 8 frames e 16 passos, resolução 720x480
def generate_video(image_path, prompt):
    image = load_image(image_path).convert("RGB").resize((720, 480))
    result = pipe(
        image=image,
        prompt=prompt,
        guidance_scale=5,
        num_inference_steps=16,
        num_frames=8
    )
    frames = result.frames[0]
    out = "outputs/cogvideo_reduced.mp4"
    export_to_video(frames, out, fps=6)
    return out

In [ ]:
# 🖼️ Interface Gradio
with gr.Blocks() as demo:
    gr.Markdown("## CogVideoX (ultra leve - 8 frames, 16 steps)")
    with gr.Row():
        img = gr.Image(type="filepath", label="Imagem")
        prm = gr.Textbox(label="Prompt (ex: 'neve caindo à noite')")
    btn = gr.Button("🎬 Gerar")
    vid = gr.Video(label="🎞️ Vídeo")
    btn.click(fn=generate_video, inputs=[img, prm], outputs=vid)
    demo.launch(share=True)